# 03 — Feature Engineering

This notebook transforms raw data into model-ready features.
Every decision made here is documented — what was done, why, and what the alternative was.

Inputs:  data/raw/adult.data, adult.test
Outputs: data/processed/train.csv, data/processed/test.csv

In [1]:
import pandas as pd
import numpy as np
import yaml
import os

In [2]:
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [3]:
columns = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]

train = pd.read_csv(
    "../data/raw/adult.data",
    names=columns,
    skipinitialspace=True
)

test = pd.read_csv(
    "../data/raw/adult.test",
    names=columns,
    skipinitialspace=True,
    skiprows=1
)

# replace ? with NaN
train.replace("?", np.nan, inplace=True)
test.replace("?", np.nan, inplace=True)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K.
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K.
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K.
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K.
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16276,39,Private,215419,Bachelors,13,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36,United-States,<=50K.
16277,64,NaN,321403,HS-grad,9,Widowed,NaN,Other-relative,Black,Male,0,0,40,United-States,<=50K.
16278,38,Private,374983,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50,United-States,<=50K.
16279,44,Private,83891,Bachelors,13,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40,United-States,<=50K.


In [4]:
print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

Train: (32561, 15)
Test:  (16281, 15)


## 1. Drop Redundant and Non-Predictive Features

In [5]:
# fnlwgt -- census sampling weight, not a predictive feature
# education-num -- perfect redundancy with education (proven in EDA)

drop_cols = ["fnlwgt", "education-num"]

train = train.drop(columns=drop_cols)
test = test.drop(columns=drop_cols)

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print(f"Remaining columns: {list(train.columns)}")

Train: (32561, 13)
Test:  (16281, 13)
Remaining columns: ['age', 'workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']


## 2. Handle Missing Values

In [6]:
# strategy decided in data audit:
# missing values are structural (MNAR), not random
# treat ? as meaningful "Unknown" category, do not impute

train["workclass"] = train["workclass"].fillna("Unknown")
train["occupation"] = train["occupation"].fillna("Unknown")
train["native-country"] = train["native-country"].fillna("Unknown")

test["workclass"] = test["workclass"].fillna("Unknown")
test["occupation"] = test["occupation"].fillna("Unknown")
test["native-country"] = test["native-country"].fillna("Unknown")

# verify no missing values remain
print("Missing values in train:")
print(train.isnull().sum()[train.isnull().sum() > 0])
print()
print("Missing values in test:")
print(test.isnull().sum()[test.isnull().sum() > 0])
print()
print("No missing values." if train.isnull().sum().sum() == 0 else "WARNING: missing values remain")

Missing values in train:
Series([], dtype: int64)

Missing values in test:
Series([], dtype: int64)

No missing values.


## 3. Clean Target Variable

In [7]:
# test set has trailing period in income values e.g. "<=50K." vs "<=50K"
# standardize both to match

print("Train income values:", train["income"].unique())
print("Test income values: ", test["income"].unique())

Train income values: <StringArray>
['<=50K', '>50K']
Length: 2, dtype: str
Test income values:  <StringArray>
['<=50K.', '>50K.']
Length: 2, dtype: str


### Observation — Target Variable Inconsistency

The UCI test set encodes income as `<=50K.` and `>50K.` — with a 
trailing period. This is a known artifact of the original data 
collection. Without cleaning, train and test targets would not 
match, causing silent errors in evaluation. Stripped and 
standardized before any further processing.

In [8]:
train["income"] = train["income"].str.strip().str.replace(".", "", regex=False)
test["income"] = test["income"].str.strip().str.replace(".", "", regex=False)

print()
print("After cleaning:")
print("Train:", train["income"].unique())
print("Test: ", test["income"].unique())


After cleaning:
Train: <StringArray>
['<=50K', '>50K']
Length: 2, dtype: str
Test:  <StringArray>
['<=50K', '>50K']
Length: 2, dtype: str


## 4. Capital Gain and Capital Loss Transformation

In [9]:
# from EDA: 91.7% zero values, cap at 99,999 -- two-part distribution
# strategy: binary flag for any capital activity + log transformation 
# for non-zero values

for df in [train, test]:
    # binary flags
    df["has_capital_gain"] = (df["capital-gain"] > 0).astype(int)
    df["has_capital_loss"] = (df["capital-loss"] > 0).astype(int)
    
    # log transformation
    df["capital_gain_log"] = np.log1p(df["capital-gain"])
    df["capital_loss_log"] = np.log1p(df["capital-loss"])

# drop original columns
train = train.drop(columns=["capital-gain", "capital-loss"])
test = test.drop(columns=["capital-gain", "capital-loss"])

print("New capital features:")
print(train[["has_capital_gain", "has_capital_loss", 
             "capital_gain_log", "capital_loss_log"]].describe())

New capital features:
       has_capital_gain  has_capital_loss  capital_gain_log  capital_loss_log
count      32561.000000      32561.000000      32561.000000      32561.000000
mean           0.083290          0.046651          0.734621          0.350305
std            0.276324          0.210893          2.454738          1.584581
min            0.000000          0.000000          0.000000          0.000000
25%            0.000000          0.000000          0.000000          0.000000
50%            0.000000          0.000000          0.000000          0.000000
75%            0.000000          0.000000          0.000000          0.000000
max            1.000000          1.000000         11.512925          8.379539


### Observation — Capital Feature Transformation

- **has_capital_gain:** only 8.3% of rows report any capital gain — confirms the zero-inflation finding from EDA
- **has_capital_loss:** even rarer at 4.7% — capital loss is a minority signal
- **capital_gain_log max: 11.51** — corresponds to the 99,999 cap value (log(99999+1) ≈ 11.51), now compressed into a reasonable range
- **75th percentile is zero for all four features** — the binary flags capture the most important signal: did any capital activity 
  occur at all?

The combination of binary flag + log transformation preserves both the presence signal and the magnitude signal without letting the 
capped outliers dominate the feature space.

## 5. Age Binning

In [10]:
# from EDA: age has non-linear relationship with income
# under 25 almost no high earners, peak 35-55, decline after
# binning captures this structure better than raw continuous age

bins = [0, 25, 35, 45, 55, 65, 100]
labels = ["<25", "25-35", "35-45", "45-55", "55-65", "65+"]

train["age_group"] = pd.cut(train["age"], bins=bins, labels=labels)
test["age_group"] = pd.cut(test["age"], bins=bins, labels=labels)

print(train.groupby("age_group", observed=True)["income"].apply(
    lambda x: (x == ">50K").mean()
).round(3))

age_group
<25      0.018
25-35    0.187
35-45    0.346
45-55    0.398
55-65    0.315
65+      0.201
Name: income, dtype: float64


### Observation — Age Binning

The binned income rates confirm the non-linear relationship:

- **<25:** only 1.8% high earners — entry level, students
- **25-35:** 18.7% — early career, income starting to grow
- **35-45:** 34.6% — peak earning trajectory
- **45-55:** 39.8% — highest income rate, peak seniority
- **55-65:** 31.5% — slight decline, early retirement effect
- **65+:** 20.1% — retirement age, income drops significantly

The monotonic rise and fall is clean and meaningful. Binning 
captures this structure explicitly rather than asking a linear 
model to infer it from raw age. We keep both `age` and `age_group` 
for now — tree-based models can use raw age directly, while 
logistic regression will benefit from the binned version.

## 6. Encode Target Variable

In [11]:
# binary encode target: >50K = 1, <=50K = 0

train["income"] = (train["income"] == ">50K").astype(int)
test["income"] = (test["income"] == ">50K").astype(int)

print("Train target distribution:")
print(train["income"].value_counts())
print()
print("Test target distribution:")
print(test["income"].value_counts())

Train target distribution:
income
0    24720
1     7841
Name: count, dtype: int64

Test target distribution:
income
0    12435
1     3846
Name: count, dtype: int64


### Observation — Target Encoding

Binary encoding confirmed: 0 = <=50K, 1 = >50K.
Class balance is consistent across train (75.3% / 24.7%) and 
test (76.4% / 23.6%) splits — no distributional shift in the target.

In [13]:
train.head()

,age,workclass,education,marital-status,occupation,relationship,race,sex,hours-per-week,native-country,income,has_capital_gain,has_capital_loss,capital_gain_log,capital_loss_log,age_group
0,39,State-gov,Bachelors,Never-married,Adm-clerical,Not-in-family,White,Male,40,United-States,0,1,0,7.684784,0.0,35-45
1,50,Self-emp-not-inc,Bachelors,Married-civ-spouse,Exec-managerial,Husband,White,Male,13,United-States,0,0,0,0.000000,0.0,45-55
2,38,Private,HS-grad,Divorced,Handlers-cleaners,Not-in-family,White,Male,40,United-States,0,0,0,0.000000,0.0,35-45
3,53,Private,11th,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,40,United-States,0,0,0,0.000000,0.0,45-55
4,28,Private,Bachelors,Married-civ-spouse,Prof-specialty,Wife,Black,Female,40,Cuba,0,0,0,0.000000,0.0,25-35


## 7. Save Processed Data

Categorical features are intentionally left unencoded at this stage.
Different models require different encoding strategies:
- Tree models: label encoding or raw categories
- Logistic regression: one-hot encoding
- Neural networks: embedding or one-hot

Encoding is handled inside each model pipeline in the modeling notebook.
This file represents the cleaned, transformed, human-readable 
intermediate state.

In [14]:
os.makedirs("../data/processed", exist_ok=True)

train.to_csv("../data/processed/train.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

print("Saved:")
print(f"  train: {train.shape}")
print(f"  test:  {test.shape}")
print()
print("Final columns:")
for col in train.columns:
    print(f"  {col}")

Saved:
  train: (32561, 16)
  test:  (16281, 16)

Final columns:
  age
  workclass
  education
  marital-status
  occupation
  relationship
  race
  sex
  hours-per-week
  native-country
  income
  has_capital_gain
  has_capital_loss
  capital_gain_log
  capital_loss_log
  age_group


## 8. Feature Engineering Part 1 - Summary

**Dropped:**
- `fnlwgt` — census sampling weight, not predictive
- `education-num` — perfect redundancy with `education` (proven in EDA)
- `capital-gain`, `capital-loss` — replaced by transformed versions

**Missing values:**
- `workclass`, `occupation`, `native-country` — filled with `Unknown` 
  category. Missingness is structural (MNAR), not random.

**New features created:**
- `has_capital_gain` — binary flag, captures presence of capital activity
- `has_capital_loss` — binary flag, captures presence of capital activity  
- `capital_gain_log` — log1p transformation, compresses zero-inflated 
  distribution and handles 99,999 cap
- `capital_loss_log` — same treatment as capital gain
- `age_group` — binned age capturing non-linear income relationship 
  identified in EDA

**Intentionally unchanged:**
- All categorical features left unencoded — encoding strategy 
  depends on model type and is handled inside each model pipeline

**Output:**
- `data/processed/train.csv` — 32,561 rows, 16 columns
- `data/processed/test.csv` — 16,281 rows, 16 columns

_______________________________________

## Part 2 — Additional Feature Engineering

Based on EDA findings, three additional transformations are applied:
1. Marital status simplification → binary `is_married` flag
2. Native country → regional grouping
3. Hours per week → binned categories

These reduce sparsity, capture non-linear relationships explicitly,
and are directly motivated by patterns observed in EDA.

## 9. Marital Status Simplification

In [15]:
married_categories = [
    "Married-civ-spouse", 
    "Married-AF-spouse"
]

train["is_married"] = train["marital-status"].isin(married_categories).astype(int)
test["is_married"] = test["marital-status"].isin(married_categories).astype(int)

print(train.groupby("is_married")["income"].mean().round(3))
print()
print(f"Married:     {train['is_married'].sum():,}")
print(f"Not married: {(train['is_married'] == 0).sum():,}")

is_married
0    0.065
1    0.447
Name: income, dtype: float64

Married:     14,999
Not married: 17,562


### Observation — Marital Status and Income

The simplified binary feature reveals a dramatic income gap:
- **Married:** 44.7% earn above $50K
- **Not married:** only 6.5% earn above $50K

This is a nearly 7x difference — `is_married` is one of the 
strongest single predictors in the dataset, stronger than 
most occupation categories.

However, this signal carries a fairness risk worth noting:
marital status is correlated with sex — the dataset has more 
married males than females, and the definition of "married" 
in 1994 census data reflects heteronormative household structures. 
A model heavily weighted on this feature may indirectly 
encode sex-based bias.

We keep both `marital-status` (original) and `is_married` (binary) 
— tree models will use the original, logistic regression 
will benefit from the binary flag.

## 10. Native Country → Regional Grouping

In [16]:
region_mapping = {
    "United-States": "North-America",
    "Canada": "North-America",
    "Mexico": "Latin-America",
    "Puerto-Rico": "Latin-America",
    "El-Salvador": "Latin-America",
    "Cuba": "Latin-America",
    "Jamaica": "Latin-America",
    "Dominican-Republic": "Latin-America",
    "Guatemala": "Latin-America",
    "Columbia": "Latin-America",
    "Haiti": "Latin-America",
    "Nicaragua": "Latin-America",
    "Peru": "Latin-America",
    "Ecuador": "Latin-America",
    "Honduras": "Latin-America",
    "Trinadad&Tobago": "Latin-America",
    "India": "Asia",
    "China": "Asia",
    "Japan": "Asia",
    "Philippines": "Asia",
    "Vietnam": "Asia",
    "Taiwan": "Asia",
    "Iran": "Asia",
    "Thailand": "Asia",
    "Hong": "Asia",
    "Cambodia": "Asia",
    "Laos": "Asia",
    "England": "Europe",
    "Germany": "Europe",
    "Italy": "Europe",
    "Poland": "Europe",
    "Portugal": "Europe",
    "France": "Europe",
    "Yugoslavia": "Europe",
    "Scotland": "Europe",
    "Greece": "Europe",
    "Ireland": "Europe",
    "Hungary": "Europe",
    "Holand-Netherlands": "Europe",
    "South": "Other",
    "Outlying-US(Guam-USVI-etc)": "Other",
    "Unknown": "Unknown"
}

train["native_region"] = train["native-country"].map(region_mapping).fillna("Other")
test["native_region"] = test["native-country"].map(region_mapping).fillna("Other")

print(train.groupby("native_region")["income"].mean().round(3))
print()
print(train["native_region"].value_counts())

native_region
Asia             0.307
Europe           0.292
Latin-America    0.079
North-America    0.246
Other            0.170
Unknown          0.250
Name: income, dtype: float64

native_region
North-America    29291
Latin-America     1401
Asia               671
Unknown            583
Europe             521
Other               94
Name: count, dtype: int64


### Observation — Regional Grouping and Income

Grouping 41 country categories into 6 regions reveals a clear pattern:

- **Asia (30.7%) and Europe (29.2%)** have the highest income rates — 
  likely reflecting selection bias: immigrants from these regions 
  in 1994 were disproportionately skilled workers and professionals
- **North-America (24.6%)** — baseline, matches overall dataset average
- **Unknown (25.0%)** — close to North-America baseline, suggesting 
  missing country data is not systematically linked to income
- **Latin-America (7.9%)** — significantly below average, lowest 
  among named regions
- **Other (17.0%)** — mid-range but small sample (94 instances), 
  interpret with caution

**Important caveat:**
The Asia and Europe premium likely reflects immigration selection 
effects, not regional origin causing higher income. People who 
immigrated from Asia or Europe to the US in the 1994 census 
were more likely to be on skilled worker visas or professional 
tracks. This is a confounding factor, not a causal relationship.

Reducing 41 categories to 6 regions also significantly reduces 
one-hot encoding dimensionality from 41 columns to 6 — 
a practical benefit for logistic regression.

_______________________________

## 11. Hours per Week Binning

In [17]:
hours_bins = [0, 34, 45, 60, 100]
hours_labels = ["part-time", "standard", "overtime", "extreme"]

train["hours_category"] = pd.cut(
    train["hours-per-week"],
    bins=hours_bins,
    labels=hours_labels
)

test["hours_category"] = pd.cut(
    test["hours-per-week"],
    bins=hours_bins,
    labels=hours_labels
)

print(train.groupby("hours_category", observed=True)["income"].mean().round(3))
print()
print(train["hours_category"].value_counts())

hours_category
part-time    0.069
standard     0.225
overtime     0.427
extreme      0.364
Name: income, dtype: float64

hours_category
standard     19839
overtime      6029
part-time     5583
extreme       1110
Name: count, dtype: int64


### Observation — Hours per Week Binning

**Income rate by hours category:**
- **Part-time (<35h):** 6.9% — very low, consistent with 
  low-wage or supplementary work
- **Standard (35-45h):** 22.5% — baseline, matches most workers
- **Overtime (45-60h):** 42.7% — strong signal, nearly 2x standard
- **Extreme (60h+):** 36.4% — slightly lower than overtime

**The extreme hours drop is the interesting finding:**
Working 60+ hours does not maximize income probability — overtime 
(45-60h) does. This could reflect:
- Self-employed or gig workers logging extreme hours at lower income
- Diminishing returns on hours beyond a threshold
- A small sample effect (only 1,110 instances in extreme category)

**Distribution is heavily concentrated in standard hours** — 
19,839 instances (60.9%). Part-time and extreme are minority 
groups that may need careful handling in modeling.

We keep both `hours-per-week` (continuous) and `hours_category` 
(binned) — same rationale as age. Tree models use continuous, 
logistic regression benefits from explicit bins.

### Save the data

In [18]:
train.to_csv("../data/processed/train.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

print("Saved:")
print(f"  train: {train.shape}")
print(f"  test:  {test.shape}")
print()
print("Final columns:")
for col in train.columns:
    print(f"  {col}")

Saved:
  train: (32561, 19)
  test:  (16281, 19)

Final columns:
  age
  workclass
  education
  marital-status
  occupation
  relationship
  race
  sex
  hours-per-week
  native-country
  income
  has_capital_gain
  has_capital_loss
  capital_gain_log
  capital_loss_log
  age_group
  is_married
  native_region
  hours_category


## Feature Engineering Complete — Final Feature Set

**Original:** 15 columns → **Final:** 19 columns

**All transformations motivated by EDA findings:**

| Feature | Type | Source |
|---|---|---|
| age | continuous | original |
| age_group | categorical | binned from age |
| workclass | categorical | original, Unknown filled |
| education | categorical | original |
| marital-status | categorical | original |
| is_married | binary | simplified from marital-status |
| occupation | categorical | original, Unknown filled |
| relationship | categorical | original |
| race | categorical | original |
| sex | categorical | original |
| hours-per-week | continuous | original |
| hours_category | categorical | binned from hours-per-week |
| native-country | categorical | original, Unknown filled |
| native_region | categorical | grouped from native-country |
| has_capital_gain | binary | from capital-gain |
| has_capital_loss | binary | from capital-loss |
| capital_gain_log | continuous | log1p of capital-gain |
| capital_loss_log | continuous | log1p of capital-loss |
| income | binary | target, encoded 0/1 |